In [16]:
import os
import sys

# Confirm vision-common actually mounted, and under what exact folder name -
# Kaggle sometimes normalizes the slug slightly differently than you typed.
print(os.listdir('/kaggle/input'))
sys.path.append('/kaggle/input/datasets/maryclauds/vision-common/vision_common')

from vision_common import detect_marker, extract_shape_features


['datasets']


### Train dir

In [21]:
train_dir = '/kaggle/input/datasets/maryclauds/good-dataset/good_dataset'

## The dataset file names & structure

In [15]:
"""
Run this in Kaggle against your dataset folder. It shows every unique
"raw label" the filename regex is actually extracting, and how many files
fall under each - so we can see exactly why star/hexagon aren't matching
LABEL_MAP without guessing.
"""

import os
import re
from collections import Counter

train_dir = '/kaggle/input/datasets/maryclauds/good-dataset/good_dataset'

FILENAME_PATTERN = re.compile(
    r'^capture_(?P<label>.+)_(?P<idx>\d+)\.(jpg|jpeg|png)$',
    re.IGNORECASE
)

raw_labels = Counter()
unmatched = []

for filename in os.listdir(train_dir):
    m = FILENAME_PATTERN.match(filename)
    if not m:
        unmatched.append(filename)
        continue
    raw_labels[m.group('label').lower()] += 1

print("=== Raw label -> file count ===")
for label, count in raw_labels.most_common():
    print(f"  '{label}': {count} files")

if unmatched:
    print(f"\n=== {len(unmatched)} files didn't match the pattern at all (showing up to 10) ===")
    for f in unmatched[:10]:
        print("  ", f)

=== Raw label -> file count ===
  'star': 200 files
  'square': 200 files
  'hex': 200 files
  'aruco_3_shapes': 30 files
  'aruco': 30 files


In [22]:
"""
Loader for the clean dataset: capture_stars_N.jpg, capture_square_N.jpg,
capture_hexagon_N.jpg (200 each, independently captured), plus
capture_arcuo_N.jpg (marker only, 50) and capture_arcuo_3_shapes_N.jpg
(marker + all 3 shapes, 30) - excluded from training, see docstring below.
"""

import re
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import pandas as pd

FILENAME_PATTERN = re.compile(
    r"^capture_(?P<label>.+)_(?P<idx>\d+)\.(jpg|jpeg|png)$",
    re.IGNORECASE
)

LABEL_MAP = {
    "star": "Star",
    "square": "Square",
    "hex": "Hexagon",
}


def load_labeled_dataset(folder_path):
    X, y = [], []
    skipped_not_single_shape = 0
    skipped_no_contour = 0

    for filename in os.listdir(folder_path):
        m = FILENAME_PATTERN.match(filename)
        if not m:
            continue

        raw_label = m.group("label").lower()
        label = LABEL_MAP.get(raw_label)
        if label is None:
            skipped_not_single_shape += 1
            continue

        image_path = os.path.join(folder_path, filename)
        img = cv2.imread(image_path)
        if img is None:
            continue

        marker = detect_marker(img)
        features, _ = extract_shape_features(img, exclude_bbox=marker["bbox"] if marker else None)

        if features is None:
            skipped_no_contour += 1
            continue

        X.append(features)
        y.append(label)

    print(f"Loaded {len(X)} labeled samples.")
    print(f"Skipped {skipped_not_single_shape} marker-only / multi-shape files.")
    print(f"Skipped {skipped_no_contour} labeled files with no valid contour found - spot-check these.")

    for shape in sorted(set(y)):
        print(f"  {shape}: {y.count(shape)} samples")

    return np.array(X), np.array(y)


## Load the data


In [23]:
X, y = load_labeled_dataset(train_dir)


Loaded 593 labeled samples.
Skipped 60 marker-only / multi-shape files.
Skipped 7 labeled files with no valid contour found - spot-check these.
  Hexagon: 200 samples
  Square: 197 samples
  Star: 196 samples


## Stratified 80/20 split

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")


Train: 474  Test: 119


## Train


In [25]:
clf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

     Hexagon       1.00      1.00      1.00        40
      Square       1.00      1.00      1.00        40
        Star       1.00      1.00      1.00        39

    accuracy                           1.00       119
   macro avg       1.00      1.00      1.00       119
weighted avg       1.00      1.00      1.00       119



## confirm the results

In [26]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    X, y, cv=5
)
print(scores)
print(f"Mean: {scores.mean():.3f}  Std: {scores.std():.3f}")

[1. 1. 1. 1. 1.]
Mean: 1.000  Std: 0.000


## Confusion matrix

In [27]:
labels = sorted(np.unique(y_test))
cm = confusion_matrix(y_test, y_pred, labels=labels)
print(pd.DataFrame(cm, index=[f"actual_{l}" for l in labels], columns=[f"pred_{l}" for l in labels]))


                pred_Hexagon  pred_Square  pred_Star
actual_Hexagon            40            0          0
actual_Square              0           40          0
actual_Star                0            0         39


## Save the model for `main_vision.py`

In [28]:
joblib.dump(clf, "shape_classifier.pkl")
print("Saved shape_classifier.pkl")


Saved shape_classifier.pkl
